In [1]:
import os
from datetime import datetime, timedelta
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import logging
import tqdm

import logger
from pats_service import PatsService
from dotenv import load_dotenv

from plot_examples import ExamplePlots

logger.init_logger(logger=logger.logger)
load_dotenv()


# Local Import
from main import read_credentials

# Notebook for downloading section information where certain moths are active

To run the code multiple python packages are necessary. So before running this notebook setup a local python environment by opening the terminal and typing the following commands:
```
python3 -m venv .venv
source .venv/bin/activate
python3 -m pip install -r requirements.txt
```

Furthermore, to run the code, select each cell and press ctrl+Enter or 

**press then run all button on top**


In [2]:
user, passw = read_credentials()

# Initialize "patsService" and "examplePlots" classes.
pats_service = PatsService(user=user, passw=passw)

logging.getLogger(name="log").setLevel(logging.ERROR)

# with patsc.open_meta_db() as con:
#     data_fd = pd.read_sql(
#         find_active_sections_turkse,
#         con,
#     )
#     sections_to_look_for = data_fd["section_id"].unique()
#     print(sections_to_look_for)


sections = pats_service.download_sections()
sections_dataframe = pd.DataFrame.from_records(sections)

def find_available_insect(detection_classes, insect_id):
    for detection_class in detection_classes:
        if detection_class['available_in_c'] and detection_class['id'] == insect_id:
            return True
    return False

INSECT_ID_WANTED = 1

sections_dataframe['insect_available'] = sections_dataframe['detection_classes'].apply(lambda x: find_available_insect(x, INSECT_ID_WANTED))

# Filter out sections that do not have the insect we are looking for
sections_dataframe = sections_dataframe[sections_dataframe['insect_available']].loc[:, ['customer_name', 'id', 'label', 'timezone', 'greenhouse_name']]


* INFO  - Successfully retrieved API token from server: https://pats-c.com/ - pats_service.60
* INFO  - Successfully retrieved API token from server: https://pats-c.com/ - pats_service.60


In [3]:
sections_to_look_for = sections_dataframe['id'].values

counts_list = []
for section in tqdm.tqdm(sections_to_look_for):
    try:
        counts = pats_service.download_counts(
            start_date=datetime.today() - timedelta(days=3),
            end_date=datetime.today(),
            section_id=section,
            detection_class_ids=[1],
        )
    # Check for when there are no turkse moth counts
    except Exception as e:
        print(e)
        continue

    pats_c = counts['c']

    # Check for when there are no systems
    if len(pats_c) == 0:
        continue

    # Combine counts from all systems in section
    all_counts = []
    for i, result in enumerate(pats_c):
        counts = result['counts']
        for count in counts:
            count['i'] = i
        all_counts.extend(counts)

    counts = pd.DataFrame(all_counts)

    # Count average number of insects per day
    counts_df = counts.groupby(['date']).agg({'1': 'mean'}).reset_index()

    # Download spots and mean latitude and longitude
    spots = pats_service.download_spots(section_id=section)
    spots_df = pd.DataFrame.from_records(spots['c'])
    latitude = spots_df['latitude'].dropna().mean()
    longitude = spots_df['longitude'].dropna().mean()

    # Fill dataframe with section_id, latitude and longitude
    counts_df['latitude'] = latitude
    counts_df['longitude'] = longitude
    counts_df['section_id'] = section

    counts_list.append(counts_df)

counts_df = pd.concat(counts_list)
display(counts_df)


100%|██████████| 173/173 [05:40<00:00,  1.97s/it]


,date,1,latitude,longitude,section_id
0,20240929,1.0,42.705190,2.817900,185
1,20240930,0.0,42.705190,2.817900,185
2,20241001,0.0,42.705190,2.817900,185
0,20240929,0.0,47.480380,7.590986,40
1,20240930,0.5,47.480380,7.590986,40
...,...,...,...,...,...
1,20240930,0.0,51.413893,6.265070,293
2,20241001,0.0,51.413893,6.265070,293
0,20240929,0.0,51.412771,6.265393,309
1,20240930,0.0,51.412771,6.265393,309


In [4]:
import math

def haversine(lat1, lon1, lat2=51.99675359068537, lon2=4.385069255656428):
    # Radius of Earth in kilometers
    R = 6371.0
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    # Difference in coordinates
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    # Haversine formula
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    # Distance in kilometers
    distance = R * c
    return distance

counts_df['distance'] = counts_df.apply(lambda x: haversine(x['latitude'], x['longitude']), axis=1)
counts_df.rename(columns={'1': 'count'}, inplace=True)

In [5]:
# Select wanted columns
counts_df_sel = counts_df[['section_id', 'count', 'distance', 'latitude', 'longitude']]

# Sort section_ids based on counts
counts_df_sel = counts_df_sel.groupby('section_id').agg({'count': 'sum', 'distance': 'mean', 'latitude': 'last', 'longitude': 'last'}).reset_index().sort_values(by='count', ascending=False)

# Display the top 20 sections
display(counts_df_sel.head(20))

# Save to csv
counts_df_sel.to_csv('counts_df_sel.csv', index=False)

,section_id,count,distance,latitude,longitude
32,79,133.000000,86.399437,51.976323,5.646341
120,311,108.000000,0.026426,51.996988,4.385136
4,14,80.500000,12.820535,51.968246,4.203680
1,2,62.000000,NaN,NaN,NaN
58,132,56.000000,880.565721,44.789807,-0.574403
9,27,43.000000,9.545883,52.004448,4.246188
77,179,37.500000,11.407115,51.986910,4.219240
0,1,31.000000,13.117986,52.013347,4.195333
24,57,27.000000,15.649966,52.036071,4.604654
52,120,25.700000,13.117986,52.013347,4.195333
